In [1]:
import os # For folder navigation
import numpy as np # For math calculations
from PIL import Image, ImageFilter # For image manipulation

In [2]:
# Define our input and output folders
base_dir = r'C:\Users\vinuga janandith\Desktop\AIML project\data'
output_dir = r'C:\Users\vinuga janandith\Desktop\AIML project\results\outputs'

In [3]:
# Member 4's Outlier Boundaries
LOWER_BOUND = 2.56
UPPER_BOUND = 235.13

In [4]:
# Member 2's Target Size
TARGET_SIZE = (128, 128)

In [5]:
print("Starting the Pipeline...")
print("Cleaning, resizing, and saving images....")

processed_count = 0
dropped_count = 0

# Loop through the dataset
for root, dirs, files in os.walk(base_dir):
    if not dirs:
        class_name = os.path.basename(root)
        
        # Create a matching subfolder in the output directory
        class_out_dir = os.path.join(output_dir, class_name)
        os.makedirs(class_out_dir, exist_ok=True)
        
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                file_path = os.path.join(root, file)
                
                try:
                    # MEMBER 1: Integrity Check (Attempt to open)
                    with Image.open(file_path) as img:
                        
                        # Convert to standard RGB to prevent color channel errors
                        img = img.convert('RGB')
                        
                        # MEMBER 4: Outlier Removal (Check brightness)
                        img_array = np.array(img)
                        brightness = np.mean(img_array)
                        
                        if brightness < LOWER_BOUND or brightness > UPPER_BOUND:
                            dropped_count += 1
                            continue # Skip this image, it's an outlier!
                        
                        # MEMBER 2: Standardization (Resize to 128x128)
                        img_resized = img.resize(TARGET_SIZE, Image.Resampling.LANCZOS)
                        
                        # MEMBER 3 & 5: (Implicitly handled later when loading images as arrays for model training)
                        
                        # Save the perfect, clean image to the new output folder
                        save_path = os.path.join(class_out_dir, f"clean_{file}")
                        img_resized.save(save_path)
                        
                        processed_count += 1
                        
                except Exception:
                    # Catch any completely corrupted files
                    dropped_count += 1 

print("-" * 40)
print("PIPELINE COMPLETE!")
print(f"Successfully cleaned and saved: {processed_count} images")
print(f"Dropped (Corrupted/Outliers): {dropped_count} images")
print("-" * 40)

Starting the Pipeline...
Cleaning, resizing, and saving images....
----------------------------------------
PIPELINE COMPLETE!
Successfully cleaned and saved: 9629 images
Dropped (Corrupted/Outliers): 31 images
----------------------------------------


In [6]:
import os # For folder navigation
import random # To randomly select images and augmentation techniques
from PIL import Image, ImageEnhance # For applying the augmentations

output_dir = r'C:\Users\vinuga janandith\Desktop\AIML project\results\outputs'

# Step 1: Figure out exactly how many images each folder currently has
class_counts = {}
for folder in os.listdir(output_dir):
    folder_path = os.path.join(output_dir, folder)
    if os.path.isdir(folder_path):
        images = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        class_counts[folder] = len(images)

# target is the folder with the absolute most images (Potholes at 3,331)
target_count = max(class_counts.values())

print("-" * 40)
print(f"Target count for a perfectly balanced dataset: {target_count} images per folder.")
print("-" * 40)
print("Generating new images... \n")

# Step 2: Augment the smaller folders until they reach the target
for folder, current_count in class_counts.items():
    if current_count < target_count:
        folder_path = os.path.join(output_dir, folder)
        
        # Grab a list of the current clean images to use as base templates
        images = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        needed_images = target_count - current_count
        
        print(f"Augmenting '{folder}': Generating {needed_images} new images...")
        
        for i in range(needed_images):
            # Pick a random existing image from this folder to augment
            src_image_name = random.choice(images)
            src_image_path = os.path.join(folder_path, src_image_name)
            
            try:
                with Image.open(src_image_path) as img:
                    # Randomly choose ONE augmentation technique to apply so they all look different
                    aug_type = random.choice(['flip', 'rotate', 'brighten'])
                    
                    if aug_type == 'flip':
                        new_img = img.transpose(Image.FLIP_LEFT_RIGHT)
                    elif aug_type == 'rotate':
                        # Rotate by a random slight angle between -15 and 15 degrees
                        angle = random.uniform(-15, 15)
                        new_img = img.rotate(angle)
                    else:
                        # Randomly adjust brightness to simulate different times of day
                        factor = random.uniform(0.7, 1.4) 
                        new_img = ImageEnhance.Brightness(img).enhance(factor)
                    
                    # Save the new augmented image with an 'aug_' prefix
                    new_img_name = f"aug_{i}_{src_image_name}"
                    new_img.save(os.path.join(folder_path, new_img_name))
            except Exception:
                pass

print("\n" + "-" * 40)
print("DATASET BALANCING COMPLETE! All folders are now equal.")
print("-" * 40)

----------------------------------------
Target count for a perfectly balanced dataset: 3331 images per folder.
----------------------------------------
Generating new images... 

Augmenting 'Broken Road Sign Issues': Generating 1539 new images...
Augmenting 'Damaged Road issues': Generating 2663 new images...
Augmenting 'Illegal Parking Issues': Generating 3227 new images...
Augmenting 'Littering Garbage on Public Places Issues': Generating 1912 new images...
Augmenting 'Mixed Issues': Generating 3143 new images...
Augmenting 'Vandalism Issues': Generating 1204 new images...

----------------------------------------
DATASET BALANCING COMPLETE! All folders are now equal.
----------------------------------------
